# 3. Which edges actually differ

Two subtype networks always disagree. The question is whether they disagree more than two
random halves of the same patients would, and that needs a null which repeats the whole
averaging procedure, not just a single network.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src.config import Config
from src.io_utils import data_path, read_matrix, read_table, results_path

cfg = Config.load("../config.json")

In [ ]:
import json
from itertools import combinations

from src.preprocessing import split_by_group
from src.rewiring import (
    differential_edges, gene_rewiring, permutation_null, rewiring_summary, significant_edges,
)

with open(results_path("networks", "run_info.json")) as fh:
    run = json.load(fh)

expr = read_matrix(results_path("preprocessed", "tcga_expression.tsv.gz"))
meta = read_table(results_path("preprocessed", "tcga_meta.tsv"), index_col=0)
groups = split_by_group(expr, meta["subtype"], cfg.subtypes, cfg.min_subtype_n)
vims = {name: read_matrix(results_path("networks", f"{name}_vim.tsv.gz")) for name in groups}
run

## A first look: how much do the top edges overlap at all

In [ ]:
rewiring_summary(vims, cfg.top_edges)

## The null

`n_permutations * 2 * n_match_reps` GENIE3 fits. It is the expensive part of the pipeline
and the reason the results are believable.

In [ ]:
null = permutation_null(
    groups, n_match=run["n_match"], n_reps=run["n_reps"], n_perm=cfg.n_permutations,
    seed=cfg.seed + 1, verbose=True, regulators=list(expr.index),
    tree_method=run["tree_method"], n_trees=run["n_trees"],
    max_features=run["max_features"], n_jobs=cfg.n_jobs,
)
null.shape

In [ ]:
a, b = sorted(vims)[0], sorted(vims)[1]
diff = differential_edges(vims[a], vims[b], null, a, b, seed=cfg.seed)
hits = significant_edges(diff, cfg.fdr)
print(len(hits), "edges at FDR", cfg.fdr)
hits.head(10)

A quick sanity check: the p-values should be close to uniform across the bulk of the
edges. A spike at zero with nothing elsewhere usually means the null is too narrow.

In [ ]:
pd.Series(np.histogram(diff["p_value"], bins=10, range=(0, 1))[0],
          index=[f"{i/10:.1f}-{(i+1)/10:.1f}" for i in range(10)])

In [ ]:
gene_rewiring(hits).head(15)